In [18]:
!pip install -q timm

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import seaborn as sns
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [19]:
DATASET_PATH = Path("/kaggle/input/datasets/naumisharanyatirth/human-embryo-dataset/embryo_dataset")
ANNOT_PATH   = Path("/kaggle/input/datasets/naumisharanyatirth/human-embryo-dataset/embryo_dataset_annotations")

SEQ_LEN = 10   
MAX_VIDEOS = 200
IMG_SIZE = 224
NUM_CLASSES = 16

In [20]:
def extract_frame_num(path):
    match = re.search(r'(\d+)', Path(path).stem)
    return int(match.group(1)) if match else -1

In [21]:
def load_videos():
    videos = {}

    for folder in DATASET_PATH.iterdir():
        if not folder.is_dir():
            continue

        csv_file = ANNOT_PATH / f"{folder.name}_phases.csv"
        if not csv_file.exists():
            continue

        df = pd.read_csv(csv_file)

        target_col = None
        for col in df.columns:
            try:
                pd.to_numeric(df[col], errors="raise")
                target_col = col
                break
            except:
                continue

        if target_col is None:
            continue

        df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

        frames = sorted(folder.glob("*.jpeg"), key=lambda x: extract_frame_num(x))

        video_data = []

        for i in range(min(len(frames), len(df))):
            label = df.iloc[i][target_col]
            if pd.isna(label):
                continue

            label = int(label)
            label = max(0, min(label, NUM_CLASSES-1))

            video_data.append((str(frames[i]), label))

        if len(video_data) > SEQ_LEN:
            videos[folder.name] = video_data

    print("Total videos:", len(videos))
    return dict(list(videos.items())[:MAX_VIDEOS])

videos = load_videos()

Total videos: 452


In [22]:
video_names = list(videos.keys())

train_vid, temp_vid = train_test_split(video_names, test_size=0.3, random_state=42)
val_vid, test_vid = train_test_split(temp_vid, test_size=0.5, random_state=42)

In [23]:
def build_sequences(video_list):
    X, y = [], []

    for vid in video_list:
        frames = videos[vid]

        for i in range(len(frames) - SEQ_LEN + 1):
            seq = frames[i:i+SEQ_LEN]

            paths = [x[0] for x in seq]
            label = seq[-1][1]

            X.append(paths)
            y.append(label)

    return X, y

X_train, y_train = build_sequences(train_vid)
X_val, y_val = build_sequences(val_vid)
X_test, y_test = build_sequences(test_vid)

In [24]:
counts = Counter(y_train)

valid_classes = [c for c in counts if counts[c] > 20]

def filter_data(X, y):
    X_new, y_new = [], []
    for i in range(len(y)):
        if y[i] in valid_classes:
            X_new.append(X[i])
            y_new.append(y[i])
    return X_new, y_new

X_train, y_train = filter_data(X_train, y_train)

In [25]:
model_cnn = timm.create_model("mobilenetv2_100", pretrained=True, num_classes=0)
model_cnn.train().to(device)   

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [26]:
def extract_features(sequences):
    feats = []

    for seq in sequences:
        imgs = []
        for p in seq:
            img = Image.open(p).convert("RGB")
            img = transform(img)
            imgs.append(img)

        imgs = torch.stack(imgs).to(device)
        f = model_cnn(imgs)

        feats.append(f.detach().cpu())

    return torch.stack(feats)

print("Extracting features...")
X_train_feat = extract_features(X_train)
X_val_feat   = extract_features(X_val)
X_test_feat  = extract_features(X_test)

Extracting features...


In [27]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [28]:
train_loader = DataLoader(SeqDataset(X_train_feat, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(SeqDataset(X_val_feat, y_val), batch_size=32)
test_loader  = DataLoader(SeqDataset(X_test_feat, y_test), batch_size=32)

In [29]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(1280, 256, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(256, NUM_CLASSES)

    def forward(self, x):
        out,_ = self.lstm(x)
        return self.fc(out[:,-1,:])

In [30]:
class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)

        ce_loss = self.ce(logits, targets)

        idx = torch.arange(logits.size(1)).float().to(device)
        pred_rank = torch.sum(probs * idx, dim=1)
        true_rank = targets.float()

        ord_loss = torch.mean((pred_rank - true_rank)**2)

        return ce_loss + 0.3 * ord_loss   

In [36]:
model = LSTMModel().to(device)
criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_val = 1e9
patience = 4
counter = 0

for epoch in range(200):
    model.train()
    train_loss = 0

    for x,y in train_loader:
        x,y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            val_loss += criterion(model(x), y).item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}: Train={train_loss/len(train_loader):.4f} Val={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "best.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

Epoch 1: Train=16.9959 Val=12.6725
Epoch 2: Train=7.3193 Val=1.8482
Epoch 3: Train=0.6556 Val=0.0801
Epoch 4: Train=0.0371 Val=0.0135
Epoch 5: Train=0.0099 Val=0.0068
Epoch 6: Train=0.0061 Val=0.0049
Epoch 7: Train=0.0046 Val=0.0040
Epoch 8: Train=0.0038 Val=0.0033
Epoch 9: Train=0.0032 Val=0.0029
Epoch 10: Train=0.0028 Val=0.0025
Epoch 11: Train=0.0024 Val=0.0022
Epoch 12: Train=0.0022 Val=0.0020
Epoch 13: Train=0.0020 Val=0.0018
Epoch 14: Train=0.0018 Val=0.0016
Epoch 15: Train=0.0016 Val=0.0015
Epoch 16: Train=0.0015 Val=0.0014
Epoch 17: Train=0.0014 Val=0.0013
Epoch 18: Train=0.0013 Val=0.0012
Epoch 19: Train=0.0012 Val=0.0011
Epoch 20: Train=0.0011 Val=0.0010
Epoch 21: Train=0.0010 Val=0.0010
Epoch 22: Train=0.0010 Val=0.0009
Epoch 23: Train=0.0009 Val=0.0009
Epoch 24: Train=0.0009 Val=0.0008
Epoch 25: Train=0.0008 Val=0.0008
Epoch 26: Train=0.0008 Val=0.0007
Epoch 27: Train=0.0008 Val=0.0007
Epoch 28: Train=0.0007 Val=0.0007
Epoch 29: Train=0.0007 Val=0.0006
Epoch 30: Train=0.000